# 02. Preprocessing & Feature Engineering

This notebook prepares the data pipeline prior to training:
- Feature engineering (`Log_Amount`, `Is_High_Amount`)
- Stratified 80/20 train/test split
- `StandardScaler` fitted strictly on training data to prevent leakage
- **SMOTE** oversampling on the training set
- Feature selection via Random Forest `SelectFromModel` median thresholding

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE

os.makedirs("../reports", exist_ok=True)
data = pd.read_csv("../data/creditcard.csv")

## 1. Feature Engineering & Stratified Split

In [ ]:
def split_data(data):
    df_engineered = data.copy()
    df_engineered["Log_Amount"] = np.log1p(df_engineered["Amount"])
    df_engineered["Is_High_Amount"] = (df_engineered["Amount"] >= 100).astype(int)
    X = df_engineered.drop("Class", axis=1)
    y = df_engineered["Class"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_data(data)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 2. Feature Scaling

In [ ]:
def scale_time_amount(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    cols_to_scale = ["Time", "Amount", "Log_Amount"]
    for col in cols_to_scale:
        X_train_scaled[col] = scaler.fit_transform(X_train[[col]])
        X_test_scaled[col] = scaler.transform(X_test[[col]])

    return X_train_scaled, X_test_scaled

X_train_scaled, X_test_scaled = scale_time_amount(X_train, X_test)
X_train_scaled.head()

## 3. Oversampling via SMOTE

In [ ]:
def apply_smote(X_train_scaled, y_train):
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

    print("\nOriginal training set class distribution:")
    print(y_train.value_counts())
    print("\nResampled training set class distribution (SMOTE):")
    print(y_train_res.value_counts())

    return X_train_res, y_train_res

X_train_res, y_train_res = apply_smote(X_train_scaled, y_train)

## 4. Random Forest Feature Selection

In [ ]:
def select_features_with_rf(X_train_res, y_train_res, X_test_scaled, X_train_scaled):
    rf_temp = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    rf_temp.fit(X_train_res, y_train_res)

    selector = SelectFromModel(rf_temp, threshold="median", prefit=True)
    X_train_sel = selector.transform(X_train_res)
    X_test_sel = selector.transform(X_test_scaled)
    selected_features = X_train_scaled.columns[selector.get_support()]

    print("\nSelected features:", list(selected_features))
    pd.Series(selected_features).to_csv("../reports/selected_features.csv", index=False)

    X_test_sel_df = pd.DataFrame(X_test_sel, columns=selected_features)
    return X_train_sel, X_test_sel, X_test_sel_df, selected_features

X_train_sel, X_test_sel, X_test_sel_df, selected_features = select_features_with_rf(
    X_train_res, y_train_res, X_test_scaled, X_train_scaled
)